# الدرس الخامس: منظومة الرسائل الحديثة وادارة سياق المحادثة (Message Architecture & Trimming)

## المقدمة والاهداف التعليمية
في هذا الدفتر، سنتناول بنية الرسائل الاساسية في `langchain_core.messages`، والدور الحيوي لكل نوع، بالاضافة الى كيفية تقليم وحذف الرسائل القديمة (Message Trimming) للحفاظ على حدود نافذة السياق (Context Window).

## انواع الرسائل الرئيسية في LangChain
1. `SystemMessage`: تحدد شخصية النموذج وقواعد السلوك والتعليمات الحاكمة.
2. `HumanMessage`: تمثل مدخلات واستفسارات المستخدم البشري.
3. `AIMessage`: تمثل مخرجات النموذج، وقد تحتوي على نص او استدعاءات ادوات (`tool_calls`).
4. `ToolMessage`: تحمل نتيجة تنفيذ اداة برمجية معينة، ويجب ان ترتبط برقم معرف فريد (`tool_call_id`) مطابق لما طلبه النموذج في `AIMessage`.

## الخطوة 1: استيراد كائنات الرسائل وادوات المعالجة
نقوم باستيراد انواع الرسائل ودوال التصفية والتقليم من `langchain_core.messages`.

In [ ]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
    trim_messages,
    filter_messages
)

print("Message classes successfully imported.")

## الخطوة 2: بناء محادثة كاملة توضح دورة حياة الرسائل
ننشئ تسلسلا واقعيا يوضح التفاعل بين المستخدم، النظام، النموذج، ونتائج الادوات البرمجية.

In [ ]:
messages_history = [
    SystemMessage(content="You are a data assistant specialized in customer orders."),
    HumanMessage(content="Can you look up order ORD-9921?"),
    AIMessage(
        content="",
        tool_calls=[{
            "name": "lookup_order",
            "args": {"order_id": "ORD-9921"},
            "id": "call_abc_123",
            "type": "tool_call"
        }]
    ),
    ToolMessage(
        content="Order ORD-9921: Status Shipped, Expected Arrival: 2 days.",
        tool_call_id="call_abc_123"
    ),
    AIMessage(content="Order ORD-9921 is currently Shipped and expected to arrive in 2 days.")
]

for msg in messages_history:
    print(f"Type: {msg.type:<12} | Content Preview: {str(msg.content)[:50]}")

## الخطوة 3: فلترة الرسائل باستخدام `filter_messages`
في العديد من التطبيقات، نحتاج الى استخراج رسائل المستخدم فقط او استبعاد استدعاءات الادوات الداخلية قبل التخزين في قاعدة البيانات.

In [ ]:
# استخراج رسائل المستخدم والنظام فقط
filtered = filter_messages(
    messages_history,
    include_types=["system", "human"]
)

print(f"Original count: {len(messages_history)}, Filtered count: {len(filtered)}")
for m in filtered:
    print(f"[{m.type}]: {m.content}")

## الخطوة 4: تقليم الرسائل وسياق المحادثة عبر `trim_messages`
تعد دالة `trim_messages` احد اهم التحديثات الحديثة في LangChain. تتيح لك تحديد اقصى عدد للرموز (Tokens) او عدد الرسائل، مع ضمان:
- الحفاظ على `SystemMessage` في البداية وعدم حذفها (`include_system=True`).
- عدم ترك استدعاء اداة `tool_calls` يتيما بدون رسالة `ToolMessage` التابعة له او العكس.

In [ ]:
# محاكاة سجل محادثة طويل
extended_history = [
    SystemMessage(content="You are a helpful coding tutor."),
    HumanMessage(content="What is a variable?"),
    AIMessage(content="A variable is a named storage location in memory."),
    HumanMessage(content="What is a function?"),
    AIMessage(content="A function is a reusable block of code that performs an action."),
    HumanMessage(content="What is a class?"),
    AIMessage(content="A class is a blueprint for creating objects in OOP."),
    HumanMessage(content="Can you give me an example of a class in Python?")
]

# تقليم المحادثة للاحتفاظ بآخر رسالتين مع الحفاظ على رسالة النظام
trimmed = trim_messages(
    extended_history,
    max_tokens=4,
    token_counter=len,  # عداد مبسط بعدد الرسائل للتوضيح
    strategy="last",
    start_on="human",
    include_system=True
)

print("Trimmed History Result:")
for msg in trimmed:
    print(f"[{msg.type}]: {msg.content}")